# Analysis of whole-cell patch-clamp recordings in Python

This notebook uses the [pyabf](https://github.com/swharden/pyABF) library to read pClamp '.abf' patch-clamp files, and the [IPFX](https://github.com/AllenInstitute/ipfx) library (developed by the Allen Institute for Brain Science) for the analysis of intrinsic electrophysiological features.

**Note:** IPFX requires a dedicated environment due to its specific dependencies. As of April 2025, I used Python 3.9 to avoid compatibility issues.

To create a new environment using the conda prompt (replace `wcr` with your preferred environment name):

1. `conda create -n wcr python=3.9`
2. `conda activate wcr`
3. `pip install ipfx pyabf`

With the provided **abf** files you can clone the repository and run all the cells in this notebook.

# Import the libraries

In [ ]:
# Import the packages (with versions used to create the notebook)
import os
import time
import numpy as np  # 1.21.5
import pandas as pd  # 0.25.3

# ipfx 1.0.5
from ipfx.feature_extractor import (SpikeFeatureExtractor,
                                    SpikeTrainFeatureExtractor)
# pyabf 2.2.8
import pyabf  # To import abf files

# matplotlib 3.5.1
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'  # To generate editable text in saved *.svg plots
plt.rcParams['savefig.transparent'] = True 
plt.rcParams.update({'font.family': 'Arial'})

# Scipy 1.7.3
from scipy import signal
from scipy.optimize import curve_fit
from scipy.signal import iirnotch

# Data paths

In [ ]:
# Adapt the paths to match your data structure
data_path = f'abf_files/'
traces_path = f'Traces/'
tables_path = f'Tables/'

# Create folders if they do not exist yet
os.makedirs(data_path, exist_ok=True)
os.makedirs(traces_path, exist_ok=True)
os.makedirs(tables_path, exist_ok=True)

# Action potentials

## Plote one sweep

In [ ]:
# Example loop to plot one sweep

sweep = 9
protocol = 'APs'
held_voltage = '70mV'
recording_name = 'PYR001'

# Loop through all files in data_path
for dirpath, dirnames, filenames in os.walk(data_path):
    for filename in filenames:
        
        # Check if the filename meets the search criteria
        if (filename.startswith(recording_name) 
            and protocol in filename 
            and held_voltage in filename 
            and filename.endswith('.abf')):
            
            # Extract filename without extension
            basename = os.path.splitext(filename)[0]
            
            # Load ABF file
            file_path = os.path.join(dirpath, filename)
            abf = pyabf.ABF(file_path)
            fs = abf.dataRate  # Sampling frequency

            # Low-pass filter
            b_lowpass, a_lowpass = signal.bessel(4, 2000, 'lowpass', fs=fs)

            # Create figure and axis
            fig, ax = plt.subplots(figsize=(5, 5))
            
            # Draw a scale bar (50 mV vertical)
            ax.plot([0.8, 0.8], [-50, 0], color='k', linewidth=1.5)

            # Select the sweep to plot
            abf.setSweep(sweepNumber=sweep)

            # Apply the filter to the voltage trace
            voltage_filtered = signal.filtfilt(b_lowpass, a_lowpass, abf.sweepY)

            # Plot the filtered sweep
            ax.plot(abf.sweepX, voltage_filtered, linewidth=1, color='black')

            # Adjust axes limits and labels
            ax.set_xlim(0.2, 0.9)
            ax.locator_params(axis='y', nbins=15)
            ax.set_ylabel(abf.sweepLabelY)
            ax.set_xlabel(abf.sweepLabelX)
            ax.set_clip_on(False)

            # Remove the box frame around the plot
            for spine in ax.spines.values():
                spine.set_visible(False)

            # Save figure as SVG (vector graphics)
            fig.tight_layout()
            fig_path = os.path.join(traces_path, f"{basename}_sweep{sweep+1}.svg")
            fig.savefig(fig_path, dpi=600)
            print(f'Plot saved: {fig_path}')
            plt.close()

## Plot all sweeps

This helps to get an overview of each recording.

In [ ]:
%%time

# Define protocol
protocol = "APs"
dataset = "PYR"

# Loop through all files in dataset_path
for dirpath, dirnames, filenames in os.walk(data_path):
    for filename in filenames:
        
        # Check if the filename meets the general conditions
        if (filename.startswith(dataset)
            and filename.endswith(".abf")):

                try:
                    basename = os.path.splitext(filename)[0]
                    neuron_id = filename.split("_")[0]
                    # Load the ABF file
                    file_path = os.path.join(dirpath, filename)
                    abf = pyabf.ABF(file_path)
                    fs = abf.dataRate  # Sampling frequency

                    # Bessel low-pass filter
                    # Combine filters if needed (see passive membrane properties cell)
                    b_lowpass, a_lowpass = signal.bessel(4, 2000, 'lowpass', analog=False, fs=fs)

                    # Create the figure and axes
                    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(4, 6), sharex=True,
                                                   gridspec_kw={'height_ratios': [6, 1]})

                    # Loop through each sweep and plot with vertical offset
                    for sweepNumber in abf.sweepList:
                        abf.setSweep(sweepNumber)

                        # Apply filtering and offset for the first plot
                        voltage_filtered = signal.filtfilt(b_lowpass, a_lowpass, 
                                                           abf.sweepY) + 0.025 * sweepNumber
                        
                        # Offset only for APs traces
                        if protocol in filename:
                            offset = 120 * sweepNumber
                        else:
                            offset = 0

                        # Plot the data with the calculated offset
                        ax1.plot(abf.sweepX, voltage_filtered + offset, linewidth=0.5, alpha=0.8, color='k')
                        ax2.plot(abf.sweepX, abf.sweepC, alpha=0.5, color='C1')

                    # Axes labels
                    ax1.set_ylabel(abf.sweepLabelY)
                    ax1.set_yticklabels([])  # Y-axis labels are not very useful for stacked view
                    ax2.set_xlabel(abf.sweepLabelX)
                    ax2.set_ylabel(abf.sweepLabelC)

                    # Save figure
                    fig.tight_layout()
                    fig_path = f"{traces_path}/{basename}.svg"
                    fig.savefig(fig_path, dpi=600)
                    print(f'Plot saved:{fig_path}')
                    plt.close()

                except ValueError as e:
                    print(f"Error {basename}: {e}")

## Spike train features

See the [ipfx.feature_extractor module](https://ipfx.readthedocs.io/en/latest/ipfx.html#module-ipfx.feature_extractor) and the [SpikeFeatureExtractor class](https://ipfx.readthedocs.io/en/latest/_modules/ipfx/feature_extractor.html#SpikeFeatureExtractor) for definitions and parameter values used in action potential detection. Whether you use the default values or modify them, save the values for reproducibility of your analysis.

In [ ]:
%%time 

protocol = "APs"
dataset = "PYR"

# Initialize data tables
table_stfx = pd.DataFrame()

# Timepoints for analysis
window_time_start_s = 0.1
window_time_end_s = 1.1
ahp_window_start_ms = 750
ahp_window_end_ms = 850
current_step_ms = 400

# Iterate through all files in data path
for dirpath, dirnames, filenames in os.walk(data_path):
    for filename in filenames:
        
        # If statements
        if (filename.startswith(dataset)
            and protocol in filename
            and filename.endswith(".abf")):
            
            # Paths and names (change accordingly)
            file_path = os.path.join(dirpath, filename)
            basename = os.path.splitext(filename)[0]
            neuron_id = basename.split("_")[0]
            experiment = basename.split("_")[1]
            held_voltage = basename.split("_")[2]
            
            try: 
                # Load the ABF file
                abf = pyabf.ABF(file_path)
                fs = abf.dataRate  # or 'abf.dataPointsPerMs * 1000'
         
                # Low-pass Bessel filter
                b_lowpass, a_lowpass = signal.bessel(4,     # Order of the filter
                                                     2000,  # Cutoff frequency
                                                     'lowpass', # Type of filter
                                                     analog=False,  # Analog or digital filter
                                                     norm='phase',  # Critical frequency normalization
                                                     fs=fs)  # fs: sampling frequency

                # Initialize currents list and table for stfx_results
                currents = []
                voltage_membrane_list = []  # List to store voltage_membrane for each sweep
                
                # Plots
                fig, ax = plt.subplots(figsize=(4, 6))

                try:
                    # Loop over sweeps
                    for sweep in abf.sweepList:                
                        abf.setSweep(sweep)
                        time = abf.sweepX
                        voltage = signal.filtfilt(b_lowpass, a_lowpass, abf.sweepY)
                        current = abf.sweepC

                        # Calculate voltage_membrane for this sweep
                        voltage_membrane = np.mean(voltage[int(10*abf.dataPointsPerMs):int(100*abf.dataPointsPerMs)])
                        voltage_membrane_list.append(voltage_membrane)

                        # Calculate mAHP
                        voltage_ahp_window = voltage[int(ahp_window_start_ms*abf.dataPointsPerMs):int(ahp_window_end_ms*abf.dataPointsPerMs)]
                        ahp_min_index = np.argmin(voltage_ahp_window)
                        ahp_min_index_global = ahp_min_index + int(ahp_window_start_ms * abf.dataPointsPerMs)
                        ahp_window_start = max(ahp_min_index_global - int(10 * abf.dataPointsPerMs), 0)
                        ahp_window_end = min(ahp_min_index_global + int(10 * abf.dataPointsPerMs), len(voltage))
                        mAHP = np.mean(voltage[ahp_window_start:ahp_window_end])

                        # Calculate the average current
                        current_step = current[int(current_step_ms * abf.dataPointsPerMs)]
                        currents.append(current_step)

                        # Extract spike features
                        start, end = window_time_start_s, window_time_end_s 
                        sfx = SpikeFeatureExtractor(start, end, filter=None, 
                                                    thresh_frac=0.05, min_peak=-20)
                        sfx_results = sfx.process(time, voltage, current)
                        stfx = SpikeTrainFeatureExtractor(start=start, end=end)
                        stfx_results = stfx.process(time, voltage, current, sfx_results)

                        # Additional data to the table
                        length = len(table_stfx)
                        table_stfx.loc[length, 'neuron_id'] = neuron_id
                        table_stfx.loc[length, 'filename'] = basename
                        table_stfx.loc[length, 'Experiment'] = experiment
                        table_stfx.loc[length, 'Held voltage'] = held_voltage
                        table_stfx.loc[length, 'current_step'] = current_step
                        table_stfx.loc[length, 'APs'] = stfx_results["avg_rate"]

                        # If any spikes are detected
                        if stfx_results['avg_rate'] > 0:
                            table_stfx.loc[length, 'adaptation'] = stfx_results["adapt"]
                            table_stfx.loc[length, 'Latency_s'] = stfx_results["latency"]
                            table_stfx.loc[length, 'ISI_CV'] = stfx_results["isi_cv"]
                            table_stfx.loc[length, 'ISI_mean_ms'] = stfx_results["mean_isi"] * 1000
                            table_stfx.loc[length, 'ISI_first_ms'] = stfx_results["first_isi"] * 1000
                            table_stfx.loc[length, 'mAHP'] = voltage_membrane - mAHP

                        # Voltage membrane for each trace
                        table_stfx.loc[length, 'Voltage_memb'] = voltage_membrane
                        
                        # Plot detected peaks
                        offset = 120 * sweep
                        ax.plot(abf.sweepX, voltage + offset, linewidth=0.5, alpha=0.8, color='k')

                        if not sfx_results.empty:
                            ax.plot(sfx_results['peak_t'], sfx_results['peak_v'] + offset, 
                                    marker='o', color='magenta', markersize=1, linestyle='None')
                            ax.plot(sfx_results["threshold_t"], sfx_results["threshold_v"] + offset, 
                                    marker='o', color='orange', markersize=1, linestyle='None')

                    # Save figure
                    ax.axvline(window_time_start_s, linestyle="dotted", color="gray")
                    ax.axvline(window_time_end_s, linestyle="dotted", color="gray")
                    ax.set_xlabel(abf.sweepLabelX)
                    ax.set_yticklabels([])  # labels are not very useful for stacked view
                    fig.tight_layout()
                    fig_path = (f"{traces_path}/{basename}_peaks.svg")
                    fig.savefig(fig_path, dpi=600)
                    plt.close()

                except AssertionError as e:
                    # Print the filename and error message
                    print(f"Error in: {filename}: {e}")
                    continue
            
            except ValueError as e:
                print(f"{filename}{e}")
                
# Save the results table
table_path = os.path.join(tables_path, f"{dataset}_APs_features.csv")
table_stfx.to_csv(table_path, index=False)
print(table_path)

# Passive membrane properties

## Function

Custom function to calculate RMP, capacitance, input resistance and tau.

In [ ]:
def passive_memb_properties(voltage, time, current, fs, 
                            cursor3_ms, cursor4_ms, cursor5_ms,
                            a_initial=-70, tau_initial=40, c_initial=0, 
                            cursor1_ms=5, cursor2_ms=50):
    """
    Inputs: 
    voltage = voltage trace, raw or pre-processed
    time = time trace
    current = current trace
    cursor1_ms = Time cursor position for start of baseline (ms)
    cursor2_ms = Time cursor position for end of baseline (ms)
    cursor3_ms = Time cursor position at peak of capacitive transient (ms)
    cursor4_ms = Time cursor position at end of capacitive transient (ms)
    cursor5_ms = Time cursor position at end of voltage step (ms)
    a_initial = initial estimate for exponential fitting
    tau_initial = initial estimate for exponential fitting
    c_initial = initial estimate for exponential fitting
    
    Returns dictionary with:
    rmp: Resting membrane potential
    current_step: Current step value
    voltage_delta: Delta voltage response to current step between cursor 3 and 4
    steady_voltage: Steady voltage response between cursor 4 and 5
    ir: Input resistance in MOhm
    tau: Decay time constant in ms
    r_squared: Quality of the exponential fitting
    capacitance: Capacitance of the passive membrane in pF
    """ 

    # Position of cursors in time bins 
    cursor1 = int(cursor1_ms * (fs/1000))
    cursor2 = int(cursor2_ms * (fs/1000))
    cursor3 = int(cursor3_ms * (fs/1000))
    cursor4 = int(cursor4_ms * (fs/1000))
    cursor5 = int(cursor5_ms * (fs/1000))

    # steady voltage response
    # global steady_voltage
    steady_voltage = np.average(voltage[cursor4:cursor5])
    
    # Resting membrane potential
    rmp = np.average(voltage[cursor1:cursor2])

    # Get the current step values
    current_step = current[cursor5]

    # Delta voltage response to current step
    voltage_delta = np.average(voltage[cursor4:cursor5]) - rmp

    # Input resistance
    ir = abs(voltage_delta/current_step)*1000

    # Calculate tau from decay time constant
    decay_voltage = voltage[cursor3:cursor4]
    decay_time = (time[cursor3:cursor4])*1000
    decay_voltage = decay_voltage[~np.isnan(decay_voltage)]
    decay_time = decay_time[~np.isnan(decay_time)]

    # Initial values (a, tau, c) for estimating the fitting
    a_initial = a_initial      # Initial value  
    tau_initial = tau_initial  # Estimated tau
    c_initial = c_initial      # Baseline value
    # popt: optimal values for the parameters
    # pcov: estimated covariance of popt
    popt, pcov = curve_fit(lambda t, a, tau, c: a * np.exp(-t/tau) + c, 
                           xdata = decay_time,     # x-data
                           ydata = decay_voltage,  # y-data
                           p0 = (a_initial, tau_initial, c_initial), maxfev = 5000)
    a = popt[0]
    tau = popt[1]
    c = popt[2]

    # Quality of the exponential fitting
    # RSS: Residual sum of squares
    RSS = np.sum(np.square(decay_voltage - (a * np.exp(-decay_time/tau) + c)))
    # TSS: total sum of squares
    TSS = np.sum(np.square(decay_voltage - np.mean(decay_voltage)))
    r_squared = 1 - RSS / TSS
    
    # capacitance
    capacitance = (tau/ir) * 1000  # Convert units to pF
       
    return {'rmp':rmp, 
            'current_step': current_step, 
            'voltage_delta': voltage_delta, 
            'steady_voltage': steady_voltage,
            'ir': ir, 
            'tau': tau, 
            'a': a,
            'c': c,
            'r_squared': r_squared, 
            'capacitance': capacitance}

## Plots and tables

In [ ]:
%%time

protocol = "10pA"
dataset = "PYR"

# Cursor positions in milliseconds
cursors_ms = [5, 100, 130, 400, 600]

# Initial guesses for exponential fitting parameters
a_initial = -70   # Resting membrane potential (RMP)
tau_initial = 30  # Time constant tau estimate
c_initial = 0     # Steady-state voltage

# Initialize DataFrame for results
table_memb_properties = pd.DataFrame()
pd.options.display.float_format = '{:.4f}'.format

# Walk through all files in the data path
for dirpath, dirnames, filenames in os.walk(data_path):
    for filename in filenames:
        
        # Process only relevant ABF files matching criteria
        if (filename.startswith(dataset)
            and protocol in filename
            and filename.endswith(".abf")):
            
            file_path = os.path.join(dirpath, filename)
            basename = os.path.splitext(filename)[0]
            # Change or remove this part according to your filenames
            neuron_id, experiment, held_voltage = basename.split("_")[:3]

            try:
                abf = pyabf.ABF(file_path)
                fs = abf.dataRate  # Sampling frequency
                
                # Design lowpass Bessel filter
                b_lowpass, a_lowpass = signal.bessel(
                    4, 2000, 'lowpass', analog=False, norm='phase', fs=fs)

                # Design notch filter at 50 or 60 Hz (optional)
                b_notch, a_notch = iirnotch(w0=50, Q=30, fs=fs)

                # Lists for storing sweep results
                currents = []
                steady_voltages = []
                capacitances = []
                taus = []
                rmp_values = []

                # Analyze first 6 sweeps
                for sweep in abf.sweepList[0:6]:
                    abf.setSweep(sweep)
                    voltage = signal.filtfilt(b_notch, a_notch, abf.sweepY)
                    voltage = signal.filtfilt(b_lowpass, a_lowpass, voltage)
                    time = abf.sweepX
                    current = abf.sweepC

                    # Extract passive membrane features
                    features = passive_memb_properties(
                        voltage, time, current, fs=fs,
                        cursor3_ms=cursors_ms[2],
                        cursor4_ms=cursors_ms[3],
                        cursor5_ms=cursors_ms[4],
                        a_initial=a_initial,
                        tau_initial=tau_initial,
                        c_initial=c_initial)

                    # Append features to table
                    length = len(table_memb_properties)
                    table_memb_properties.loc[length, 'neuron_id'] = neuron_id
                    table_memb_properties.loc[length, 'filename'] = basename
                    table_memb_properties.loc[length, 'Experiment'] = experiment
                    table_memb_properties.loc[length, 'Held voltage'] = held_voltage
                    table_memb_properties.loc[length, 'rmp_mV'] = features['rmp']
                    table_memb_properties.loc[length, 'current_steps_pA'] = features['current_step']
                    table_memb_properties.loc[length, 'V_delta_mV'] = features['voltage_delta']
                    table_memb_properties.loc[length, 'V_steady_mV'] = features['steady_voltage']

                    rmp_values.append(features['rmp'])

                    if features['current_step'] != 0:
                        currents.append(features['current_step'])
                        steady_voltages.append(features['steady_voltage'])
                        capacitances.append(features['capacitance'])
                        taus.append(features['tau'])

                        table_memb_properties.loc[length, 'Rin_MOhm'] = features['ir']
                        table_memb_properties.loc[length, 'tau_ms'] = features['tau']
                        table_memb_properties.loc[length, 'tau_R squared'] = features['r_squared']
                        table_memb_properties.loc[length, 'capacitance_pF'] = features['capacitance']

                # Linear regression on current vs steady voltage (input resistance)
                m, b = np.polyfit(currents, steady_voltages, 1)
                r_squared = 1 - np.sum((steady_voltages - (m * np.array(currents) + b)) ** 2) / \
                                np.sum((steady_voltages - np.mean(steady_voltages)) ** 2)

                # Store mean values for the recording
                table_memb_properties.loc[length, 'rmp_mV_mean'] = np.mean(rmp_values)
                table_memb_properties.loc[length, 'ir_I-V_MOhm'] = m * 1000  # Convert to MOhm
                table_memb_properties.loc[length, 'r2_I-V'] = r_squared
                table_memb_properties.loc[length, 'capacitance_pF_mean'] = np.mean(capacitances)
                table_memb_properties.loc[length, 'tau_ms_mean'] = np.mean(taus)

                # Plotting setup
                fig = plt.figure(figsize=(18, 5))
                plt.rcParams.update({'font.size': 12})

                # Graph 1: Filtered traces with cursor lines
                ax1 = fig.add_subplot(131)
                ax1.set_title('Graph 1: Filtered traces')
                ax1.set_xlabel('Time (ms)')
                ax1.set_ylabel('Voltage (mV)')
                ax1top = ax1.secondary_xaxis('top')
                ax1top.set_xticks(cursors_ms)
                ax1top.set_xticklabels(['1', '2', '3', '4', '5'])
                for i, cursor in enumerate(cursors_ms):
                    ax1.axvline(cursor, linestyle="dotted", color='gray')

                # Plot filtered voltage traces
                for sweep in abf.sweepList[0:6]:
                    abf.setSweep(sweep)
                    voltage = signal.filtfilt(b_notch, a_notch, abf.sweepY)
                    voltage = signal.filtfilt(b_lowpass, a_lowpass, abf.sweepY)
                    time = abf.sweepX
                    ax1.plot(time * 1000, voltage)

                # Graph 2: Voltage decay and tau fitting
                ax2 = fig.add_subplot(132)
                ax2.set_title('Graph 2: Tau fitting')
                
                for sweep in abf.sweepList[0:6]:
                    current_step = current[cursors_ms[4]*abf.dataPointsPerMs]
                    if current_step != 0: 
                        abf.setSweep(sweep)
                        voltage = signal.filtfilt(b_lowpass, a_lowpass, abf.sweepY)
                        time = abf.sweepX
                        
                        # Extract passive membrane features for plotting
                        features = passive_memb_properties(
                            voltage, time, current, fs=fs,
                            cursor3_ms=cursors_ms[2],
                            cursor4_ms=cursors_ms[3],
                            cursor5_ms=cursors_ms[4],
                            a_initial=a_initial,
                            tau_initial=tau_initial,
                            c_initial=c_initial)
                        
                        # Decay time window
                        decay_start, decay_end = cursors_ms[2], cursors_ms[3]
                        cursor3 = np.searchsorted(time * 1000, decay_start)
                        cursor4 = np.searchsorted(time * 1000, decay_end)
                        decay_voltage = voltage[cursor3:cursor4]
                        decay_time = time[cursor3:cursor4] * 1000

                        # Fit exponential
                        a = features['a']
                        tau = features['tau']
                        c = features['c']
                        v_fitted = a * np.exp(-decay_time / tau) + c
                        ax2.plot(decay_time, decay_voltage, color='gray', alpha=0.5)
                        ax2.plot(decay_time, v_fitted)

                ax2.set_xlabel('Time window (ms)')
                ax2.set_ylabel('Voltage (mV)')

                # Graph 3: I-V plot for input resistance
                ax3 = fig.add_subplot(133)
                ax3.set_title('Graph 3: I-V')

                # Filter table for the current recording
                current_recording_data = table_memb_properties[table_memb_properties['filename'] == basename]
                current = current_recording_data['current_steps_pA']
                voltage = current_recording_data['V_steady_mV']
                
                # Linear regression and R-squared 
                m, b = np.polyfit(current, voltage, 1)
                line = np.polyval([m, b], current)
                residuals = voltage - line
                RSS = np.sum(residuals ** 2)
                TSS = np.sum((voltage - np.mean(voltage)) ** 2)
                r_squared = 1 - (RSS / TSS)

                ax3.plot(current, line, color='r')
                ax3.scatter(current, voltage)
                ax3.set_xlabel('Current steps (pA)')
                ax3.set_ylabel('Voltage (mV)')

                fig.tight_layout()
                fig.savefig(f"{traces_path}{neuron_id}_tau_iv.svg", dpi=300)
                plt.close()

            except Exception as e:
                print(f"Error {filename}: {e}")

# Save results to CSV
table_passive_path = f"{tables_path}PYR_passive_memb_properties.csv"
table_memb_properties.to_csv(table_passive_path, index=False)
print(f"Table saved to: {table_passive_path}")

# Voltage sag

## Function

In [ ]:
def voltage_sag(time, current, voltage, 
                baseline_window_start=0, baseline_window_end=100, 
                sag_window_start=250, sag_window_end=400, 
                steady_window_start=600, steady_window_end=700):
    
    """
    Function to calculate voltage sag from whole-cell patch clamp recordings. 
    
    Args:
    - time: Array of time points.
    - current: Array of current measurements.
    - voltage: Array of voltage measurements.
    - baseline_window_start: Start time (ms) for the baseline voltage window.
    - baseline_window_end: End time (ms) for the baseline voltage window.
    - sag_window_start: Start time (ms) for the sag analysis window.
    - sag_window_end: End time (ms) for the sag analysis window.
    - steady_window_start: Start time (ms) for the steady voltage window.
    - steady_window_end: End time (ms) for the steady voltage window.

    Returns:
    Table with the sag analysis results.
    """
    
    # Initialize variables to avoid errors when sag peak is not found
    sag_peak_time = None
    sag_peak_index = None
    
    # Current step (you may need to change these values too)
    current_step = current[int(steady_window_end * abf.dataPointsPerMs)]
    
    # Voltage steady response
    voltage_evoked_steady = np.mean(voltage[int(steady_window_start * abf.dataPointsPerMs):int(steady_window_end * abf.dataPointsPerMs)])

    # Voltage baseline
    voltage_baseline = np.mean(voltage[int(baseline_window_start * abf.dataPointsPerMs):int(baseline_window_end * abf.dataPointsPerMs)])

    # Voltage sag peak amplitude
    sag_peak_window_index = np.argmin(voltage[int(sag_window_start * abf.dataPointsPerMs):int(sag_window_end * abf.dataPointsPerMs)])
    sag_peak_start = int(sag_peak_window_index - 10 * abf.dataPointsPerMs) + int(sag_window_start * abf.dataPointsPerMs)
    sag_peak_end = int(sag_peak_window_index + 10 * abf.dataPointsPerMs) + int(sag_window_start * abf.dataPointsPerMs)
    sag_peak_voltage = np.median(voltage[sag_peak_start:sag_peak_end])
    sag_peak_index = sag_peak_window_index + int(sag_window_start * abf.dataPointsPerMs)
    sag_peak_time = time[sag_peak_index]

    # Sag amplitude
    voltage_sag_amplitude = np.abs(sag_peak_voltage - voltage_evoked_steady)

    # Sag ratio
    sag_ratio = (voltage_sag_amplitude / (np.abs(voltage_baseline - sag_peak_voltage))) * 100

    # Return
    return {
        'current_step': current_step,
        'voltage_baseline': voltage_baseline,
        'sag_peak_voltage': sag_peak_voltage,
        'sag_peak_index': sag_peak_index,
        'sag_peak_time': sag_peak_time,
        'voltage_evoked_steady': voltage_evoked_steady,
        'voltage_sag_amplitude': voltage_sag_amplitude,
        'sag_ratio': sag_ratio}

## Plots and tables

In [ ]:
%%time

protocol = "sag"
dataset = "PYR"

# Timepoints for analysis
baseline_window_start = 0
baseline_window_end = 150 
sag_window_start = 250
sag_window_end = 400
steady_window_start = 600
steady_window_end = 700

# Initialize data table
table_sag = pd.DataFrame()

# Iterate through all files in data_path
for dirpath, dirnames, filenames in os.walk(data_path):
    for filename in filenames:
        
        # Check if the filename meets the conditions
        if (filename.startswith(dataset)
            and protocol in filename
            and filename.endswith(".abf")):
            
            file_path = os.path.join(dirpath, filename)
            basename = os.path.splitext(filename)[0]
            neuron_id, experiment, held_voltage = basename.split("_")[:3]
            
            try:
                # Load the ABF file
                abf = pyabf.ABF(file_path)
                fs = abf.dataRate  # Sampling frequency

                # Bessel lowpass filter (4th order, cutoff 2000 Hz)
                b_lowpass, a_lowpass = signal.bessel(4, 1000, 'lowpass', analog=False, fs=fs)

                # Create the figure and axes
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(4, 6), 
                                               sharex=True, gridspec_kw={'height_ratios': [2, 1]})

                for sweepNumber in abf.sweepList:
                    abf.setSweep(sweepNumber)
                    current = abf.sweepC
                    time = abf.sweepX
                    voltage = signal.filtfilt(b_lowpass, a_lowpass, abf.sweepY) 
                    
                    current_step = current[int(sag_window_end * abf.dataPointsPerMs)]
                    if current_step != 0:

                        # Initialize variables to avoid errors when sag peak is not found
                        sag_peak_time = None
                        sag_peak_index = None

                        # Get voltage sag calculations
                        sag_properties = voltage_sag(
                            time, current, voltage, 
                            sag_window_start=sag_window_start, 
                            sag_window_end=sag_window_end, 
                            steady_window_start=steady_window_start, 
                            steady_window_end=steady_window_end)

                        # Table
                        length = len(table_sag)
                        table_sag.loc[length, 'ID_neuron'] = neuron_id
                        table_sag.loc[length, 'Filename'] = basename
                        table_sag.loc[length, 'Experiment'] = experiment
                        table_sag.loc[length, 'Held_voltage'] = held_voltage
                        table_sag.loc[length, 'Current_step'] = current_step
                        table_sag.loc[length, 'Baseline_voltage'] = sag_properties['voltage_baseline']
                        table_sag.loc[length, 'Sag_peak_voltage'] = sag_properties['sag_peak_voltage']
                        table_sag.loc[length, 'Sag_peak_index'] = sag_properties['sag_peak_index']
                        table_sag.loc[length, 'Sag_peak_time'] = sag_properties['sag_peak_time']
                        table_sag.loc[length, 'Steady voltage'] = sag_properties['voltage_evoked_steady']
                        table_sag.loc[length, 'Sag_amplitude_voltage'] = sag_properties['voltage_sag_amplitude']
                        table_sag.loc[length, 'Sag_ratio'] = sag_properties['sag_ratio']

                        if sag_properties['sag_peak_index'] is not None:

                            # Plot the data the traces
                            ax1.plot(time, voltage, linewidth=0.5)
                            ax2.plot(time, current)

                            # Plot sag peaks
                            ax1.plot(sag_properties['sag_peak_time'], 
                                     voltage[sag_properties['sag_peak_index']], 'ko', markersize=2)

                # Adjust axes and save the plot
                ax1.set_ylabel(abf.sweepLabelY)
                ax1.axvline(x=sag_window_start/1000, color='grey', linestyle='--', linewidth=0.5)
                ax1.axvline(x=sag_window_end/1000, color='grey', linestyle='--', linewidth=0.5)
                ax2.set_xlabel(abf.sweepLabelX)
                ax2.set_ylabel(abf.sweepLabelC)

                # Save figure
                fig.tight_layout()
                fig.savefig(f"{traces_path}/{basename}_results.svg")
                plt.close()

            except Exception as e:
                print(f"Error processing {filename}: {e}")

# Save the table_ih DataFrame to a CSV file after all files are processed
table_sag_path = f"{tables_path}PYR_sag_results.csv"
table_sag.to_csv(table_sag_path, index=False)
print(table_sag_path)